# YOLOv8m — FieldPlant Object Detection

Training on the **FieldPlant** dataset (manhhoangvan/fieldplant on Kaggle).
Detects 27 crop disease classes across Cassava, Corn, and Tomato.

**Setup:**
1. Add dataset: `manhhoangvan/fieldplant`
2. Enable **GPU T4 x2** (or P100)
3. Run All

In [ ]:
# Cell 1 — GPU + environment check
import os
import subprocess

print('=== GPU INFO ===')
!nvidia-smi

print('\n=== KAGGLE INPUT ===')
!ls /kaggle/input/

print('\n=== DATASET STRUCTURE ===')
base = '/kaggle/input/fieldplant'
if os.path.exists(base):
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        if depth > 2:
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth == 2:
            print(f'{indent}  ... ({len(files)} files)')
else:
    print('Dataset not found at /kaggle/input/fieldplant — check ls output above.')

In [ ]:
# Cell 2 — Install ultralytics (pinned for reproducibility)
!pip install -q ultralytics==8.3.0

import ultralytics
ultralytics.checks()   # prints device, CUDA, OpenCV, torch versions

In [ ]:
# Cell 3 — Configuration
import torch
import yaml
from pathlib import Path

# ═══════════════════════════════════════════════════════
#  CONFIGURATION — adjust if Kaggle path differs
# ═══════════════════════════════════════════════════════
DATASET_ROOT = Path('/kaggle/input/fieldplant')
WORK_DIR     = Path('/kaggle/working')

MODEL        = 'yolov8m.pt'   # pretrained backbone; swap to 'yolov8l.pt' for higher accuracy
IMGSZ        = 640
EPOCHS       = 50
BATCH        = 16             # reduce to 8 if OOM on single GPU
WORKERS      = 4
PATIENCE     = 15             # early-stop if no improvement for N epochs
LR0          = 1e-2           # initial LR (YOLO default)
LRF          = 1e-2           # final LR fraction
OPTIMIZER    = 'AdamW'
DEVICE       = '0,1' if torch.cuda.device_count() > 1 else '0'
PROJECT_NAME = 'yolov8m_fieldplant'
RUN_NAME     = 'train'

print(f'Torch:   {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()} — {torch.cuda.device_count()} GPU(s)')
print(f'Device:  {DEVICE}')
print(f'Dataset: {DATASET_ROOT}')

# ═══════════════════════════════════════════
#  27-CLASS NAMES (FieldPlant Roboflow v11)
# ═══════════════════════════════════════════
CLASS_NAMES = [
    'Cassava Bacterial Blight',
    'Cassava Brown Leaf Spot',
    'Cassava Healthy',
    'Cassava Mosaic',
    'Cassava Root Rot',
    'Corn Brown Spots',
    'Corn Charcoal',
    'Corn Chlorotic Leaf Spot',
    'Corn Gray leaf spot',
    'Corn Healthy',
    'Corn Insects Damages',
    'Corn Mildew',
    'Corn Purple Discoloration',
    'Corn Smut',
    'Corn Streak',
    'Corn Stripe',
    'Corn Violet Decoloration',
    'Corn Yellow Spots',
    'Corn Yellowing',
    'Corn leaf blight',
    'Corn rust leaf',
    'Tomato Brown Spots',
    'Tomato bacterial wilt',
    'Tomato blight leaf',
    'Tomato healthy',
    'Tomato leaf mosaic virus',
    'Tomato leaf yellow virus',
]

assert len(CLASS_NAMES) == 27, f'Expected 27 classes, got {len(CLASS_NAMES)}'
print(f'\nClasses ({len(CLASS_NAMES)}): {CLASS_NAMES[:3]} ... {CLASS_NAMES[-2:]}')

In [ ]:
# Cell 4 — Write data.yaml with absolute paths
#
# The FieldPlant dataset already ships with a data.yaml, but we rewrite it
# here so it uses absolute /kaggle/input/... paths that YOLO can always find.

data_yaml = {
    'path' : str(DATASET_ROOT),
    'train': str(DATASET_ROOT / 'train' / 'images'),
    'val'  : str(DATASET_ROOT / 'valid' / 'images'),
    'test' : str(DATASET_ROOT / 'test'  / 'images'),
    'nc'   : 27,
    'names': CLASS_NAMES,
    'roboflow': {
        'workspace': 'plant-disease-detection',
        'project'  : 'fieldplant',
        'version'  : 11,
        'license'  : 'CC BY 4.0',
        'url'      : 'https://universe.roboflow.com/plant-disease-detection/fieldplant/dataset/11',
    }
}

yaml_path = WORK_DIR / 'fieldplant.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f'data.yaml written to: {yaml_path}')
print()
!cat /kaggle/working/fieldplant.yaml

# Quick sanity check — make sure image dirs actually exist
for split in ['train', 'valid', 'test']:
    p = DATASET_ROOT / split / 'images'
    imgs = list(p.glob('*.[jp][pn][g]')) if p.exists() else []
    print(f'{split:5s}: {len(imgs):5d} images  ({'OK' if imgs else 'MISSING'})')

In [ ]:
# Cell 5 — Class distribution in train set
import collections
from pathlib import Path
import matplotlib.pyplot as plt

label_dir = DATASET_ROOT / 'train' / 'labels'
counts = collections.Counter()

for lf in label_dir.glob('*.txt'):
    with open(lf) as f:
        for line in f:
            cls_id = int(line.strip().split()[0])
            counts[cls_id] += 1

print(f'Total bounding boxes in train: {sum(counts.values()):,}')
print()
for idx in sorted(counts):
    print(f'  [{idx:2d}] {CLASS_NAMES[idx]:<35s}: {counts[idx]:5d}')

# Bar chart
fig, ax = plt.subplots(figsize=(14, 5))
xs = [CLASS_NAMES[i] for i in sorted(counts)]
ys = [counts[i] for i in sorted(counts)]
bars = ax.bar(xs, ys, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Train set — Bounding box count per class', fontsize=13)
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(WORK_DIR / 'class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Cell 6 — Train YOLOv8m
from ultralytics import YOLO

model = YOLO(MODEL)   # downloads yolov8m.pt pretrained on COCO

results = model.train(
    data       = str(yaml_path),
    epochs     = EPOCHS,
    imgsz      = IMGSZ,
    batch      = BATCH,
    workers    = WORKERS,
    patience   = PATIENCE,       # early stopping
    optimizer  = OPTIMIZER,
    lr0        = LR0,
    lrf        = LRF,
    device     = DEVICE,
    project    = str(WORK_DIR / 'runs'),
    name       = PROJECT_NAME,
    exist_ok   = True,
    save        = True,          # save best.pt + last.pt
    save_period = 10,            # also checkpoint every 10 epochs
    plots       = True,          # generate training-curve PNGs
    val         = True,
    # Augmentation (sensible defaults for field imagery)
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 5.0,
    translate  = 0.1,
    scale      = 0.5,
    fliplr     = 0.5,
    flipud     = 0.1,
    mosaic     = 1.0,
    mixup      = 0.1,
    close_mosaic = 10,           # disable mosaic in last 10 epochs for stability
    verbose    = True,
)

print('\n✅ Training complete!')
print(f'Best weights: {WORK_DIR}/runs/{PROJECT_NAME}/weights/best.pt')

In [ ]:
# Cell 7 — Validation on val set (best.pt)
best_weights = WORK_DIR / 'runs' / PROJECT_NAME / 'weights' / 'best.pt'
model_best   = YOLO(str(best_weights))

val_metrics = model_best.val(
    data    = str(yaml_path),
    split   = 'val',
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = DEVICE,
    conf    = 0.001,   # low conf to not filter out TP for mAP calculation
    iou     = 0.6,
    plots   = True,
    save_json = True,  # COCO-format JSON (useful for further analysis)
)

print('\n=== VALIDATION METRICS (val set) ===')
print(f'mAP@50       : {val_metrics.box.map50:.4f}')
print(f'mAP@50-95    : {val_metrics.box.map:.4f}')
print(f'Precision    : {val_metrics.box.mp:.4f}')
print(f'Recall       : {val_metrics.box.mr:.4f}')

In [ ]:
# Cell 8 — Test set evaluation
test_metrics = model_best.val(
    data    = str(yaml_path),
    split   = 'test',
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = DEVICE,
    conf    = 0.001,
    iou     = 0.6,
    plots   = True,
    save_json = True,
)

print('\n=== TEST SET METRICS ===')
print(f'mAP@50       : {test_metrics.box.map50:.4f}')
print(f'mAP@50-95    : {test_metrics.box.map:.4f}')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')

# Per-class AP
print('\n--- Per-class AP@50 ---')
for name, ap in zip(CLASS_NAMES, test_metrics.box.ap50):
    print(f'  {name:<35s}: {ap:.4f}')

In [ ]:
# Cell 9 — Visualise training curves
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

run_dir = WORK_DIR / 'runs' / PROJECT_NAME

# YOLO saves these automatically when plots=True
plot_files = [
    ('results.png',           'Training Curves'),
    ('confusion_matrix.png',  'Confusion Matrix'),
    ('PR_curve.png',          'Precision-Recall Curve'),
    ('F1_curve.png',          'F1 Curve'),
]

for fname, title in plot_files:
    fp = run_dir / fname
    if fp.exists():
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.imshow(mpimg.imread(fp))
        ax.axis('off')
        ax.set_title(title, fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print(f'{fname} not found — skipping')

In [ ]:
# Cell 10 — Sample inference on test images
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

test_img_dir = DATASET_ROOT / 'test' / 'images'
test_imgs    = sorted(test_img_dir.glob('*.[jp][pn][g]'))
sample_imgs  = random.sample(test_imgs, min(8, len(test_imgs)))

pred_results = model_best.predict(
    source = [str(p) for p in sample_imgs],
    imgsz  = IMGSZ,
    conf   = 0.25,
    iou    = 0.45,
    device = DEVICE,
    save   = False,
    verbose= False,
)

n = len(pred_results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten() if n > 1 else [axes]

for ax, r in zip(axes, pred_results):
    annotated = r.plot()          # BGR numpy array
    ax.imshow(annotated[:, :, ::-1])
    ax.axis('off')
    n_det = len(r.boxes)
    fname = Path(r.path).name
    ax.set_title(f'{fname}\n{n_det} detection(s)', fontsize=8)

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle('Sample Test Detections (conf ≥ 0.25)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / 'sample_detections.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11 — Summary table
print('=' * 55)
print('  FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Model       : YOLOv8m')
print(f'  Dataset     : FieldPlant (Roboflow v11)')
print(f'  Classes     : {len(CLASS_NAMES)}')
print(f'  Image size  : {IMGSZ}x{IMGSZ}')
print(f'  Epochs run  : (see results.png)')
print()
print('  --- Val set ---')
print(f'  mAP@50      : {val_metrics.box.map50:.4f}')
print(f'  mAP@50-95   : {val_metrics.box.map:.4f}')
print(f'  Precision   : {val_metrics.box.mp:.4f}')
print(f'  Recall      : {val_metrics.box.mr:.4f}')
print()
print('  --- Test set ---')
print(f'  mAP@50      : {test_metrics.box.map50:.4f}')
print(f'  mAP@50-95   : {test_metrics.box.map:.4f}')
print(f'  Precision   : {test_metrics.box.mp:.4f}')
print(f'  Recall      : {test_metrics.box.mr:.4f}')
print('=' * 55)
print()
print('  COPY THIS INTO src/config.py')
print()
print(f'  YOLO_PATH: str = "models/best_yolo8m.pt"')
print(f'  YOLO_FALLBACK_MODEL: str = "yolov8m.pt"')
print('=' * 55)

In [ ]:
# Cell 12 — Package all outputs into a zip for download
import shutil
import zipfile
from pathlib import Path

output_dir = WORK_DIR / 'outputs'
output_dir.mkdir(exist_ok=True)

run_dir = WORK_DIR / 'runs' / PROJECT_NAME

# 1. Copy weights
weights_dir = output_dir / 'weights'
weights_dir.mkdir(exist_ok=True)
for wf in (run_dir / 'weights').glob('*.pt'):
    shutil.copy(wf, weights_dir / wf.name)
    print(f'Copied: {wf.name}')

# 2. Copy plots
plots_dir = output_dir / 'plots'
plots_dir.mkdir(exist_ok=True)
for pf in list(run_dir.glob('*.png')) + [WORK_DIR / 'class_distribution.png', WORK_DIR / 'sample_detections.png']:
    if Path(pf).exists():
        shutil.copy(pf, plots_dir / Path(pf).name)
        print(f'Copied: {Path(pf).name}')

# 3. Copy results CSV + JSON
for rf in list(run_dir.glob('*.csv')) + list(run_dir.glob('*.json')):
    shutil.copy(rf, output_dir / rf.name)
    print(f'Copied: {rf.name}')

# 4. Zip everything
zip_path = WORK_DIR / f'{PROJECT_NAME}_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_dir.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(output_dir))

zip_size = zip_path.stat().st_size / (1024 * 1024)
print(f'\n✅ ZIP ready: {zip_path}  ({zip_size:.1f} MB)')
print()
print('Download from the Output tab on the right →')
print('  best.pt      → place in  models/best_yolo8m.pt')
print('  plots/*.png  → check training curves & confusion matrix')